In [2]:
!pip install -q \
  transformers \
  datasets \
  peft \
  bitsandbytes \
  accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 15.9 MB/s eta 0:00:00


In [3]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)


In [4]:
MODEL_ID = "Qwen/Qwen2.5-1.5B"   # you can change
DATA_PATH = "/content/train_clean.jsonl" # cleaned dataset
OUTPUT_DIR = "outputs"
ADAPTER_DIR = "adapters"

MAX_SEQ_LEN = 512
BATCH_SIZE = 4
LR = 2e-4
EPOCHS = 3


In [5]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)


In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.model_max_length = MAX_SEQ_LEN


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16   # 🔥 CRITICAL
)


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [8]:
model = prepare_model_for_kbit_training(model)


In [9]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "v_proj"

    ]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [10]:
dataset = load_dataset(
    "json",
    data_files=DATA_PATH
)["train"]


Generating train split: 0 examples [00:00, ? examples/s]

In [11]:
def tokenize_fn(example):
    prompt = (
        f"### Instruction:\n{example['instruction']}\n\n"
        f"### Input:\n{example['input']}\n\n"
        f"### Output:\n"
    )

    full_text = prompt + example["output"]

    tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding="max_length"
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens


In [12]:
dataset = dataset.map(
    tokenize_fn,
    remove_columns=dataset.column_names
)


Map:   0%|          | 0/3600 [00:00<?, ? examples/s]

In [13]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=1,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    logging_steps=25,
    save_strategy="epoch",
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    max_grad_norm=0.0,   # 🔥 CRITICAL FIX
    report_to="none"
)


In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )
)


In [15]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
25,1.139400
50,0.797000
75,0.592500
100,0.546900
125,0.633100
150,0.575700
175,0.590500
200,0.558200
225,0.544300
250,0.569000


TrainOutput(global_step=2700, training_loss=0.5337683808362043, metrics={'train_runtime': 4075.9246, 'train_samples_per_second': 2.65, 'train_steps_per_second': 0.662, 'total_flos': 4.3546252935168e+16, 'train_loss': 0.5337683808362043, 'epoch': 3.0})

In [16]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)


('adapters/tokenizer_config.json',
 'adapters/special_tokens_map.json',
 'adapters/chat_template.jinja',
 'adapters/vocab.json',
 'adapters/merges.txt',
 'adapters/added_tokens.json',
 'adapters/tokenizer.json')

In [18]:
!zip -r adapters.zip adapters
!zip -r outputs.zip outputs

updating: adapters/ (stored 0%)
updating: adapters/adapter_config.json (deflated 57%)
updating: adapters/merges.txt (deflated 57%)
updating: adapters/vocab.json (deflated 61%)
updating: adapters/README.md (deflated 65%)
updating: adapters/chat_template.jinja (deflated 71%)
updating: adapters/tokenizer.json (deflated 81%)
updating: adapters/tokenizer_config.json (deflated 89%)
updating: adapters/added_tokens.json (deflated 67%)
updating: adapters/adapter_model.safetensors (deflated 8%)
updating: adapters/special_tokens_map.json (deflated 62%)
  adding: outputs/ (stored 0%)
  adding: outputs/checkpoint-1800/ (stored 0%)
  adding: outputs/checkpoint-1800/adapter_config.json (deflated 57%)
  adding: outputs/checkpoint-1800/merges.txt (deflated 57%)
  adding: outputs/checkpoint-1800/trainer_state.json (deflated 85%)
  adding: outputs/checkpoint-1800/vocab.json (deflated 61%)
  adding: outputs/checkpoint-1800/README.md (deflated 65%)
  adding: outputs/checkpoint-1800/scheduler.pt (deflated 6

In [19]:
from google.colab import files
files.download("adapters.zip")
files.download("outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>